# MOSAIC Case Format Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 – Overview | This document | — |
| 2 – Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 – Config | Parameters / widgets | — |
| 4 – Constants | Case type taxonomy, column patterns, AI prompts, standardization rules | §1, §2 |
| 5 – Discovery | Table & column inventory from information_schema | — |
| 6 – Profiler | Sample-based stats: case compliance rate, violation count, sample bad values | §1, §5 |
| 7 – AI Classifier | Classify each STRING column into exactly one case type; flag ambiguous fields | §1, §3 |
| 8 – Rule Engine | Evaluate compliance against classified case type; fire violations | §1, §2, §5 |
| 9 – Write Results | Append findings to results Delta table | — |
| 10 – Summary | Blocking defects first, then ranked tables and columns | §5, Enforcement |

> **Hard constraint:** `UPDATE`, `MERGE`, `DELETE`, `CTAS` on source tables are never executed.
> **Scope:** All STRING / VARCHAR / CHAR columns are screened. AI classifies each into the procedure taxonomy; deterministic rules evaluate compliance.
> **AI use:** `ai_query()` classifies each column's **case type** (the single rule from §1 that applies to it). Deterministic SQL then checks whether actual values comply with that rule. Classification confidence is recorded; low-confidence columns are flagged for steward review.


# MOSAIC Case Format Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## Case types (§1 taxonomy)
| Case type key | Rule | Applies to |
|---|---|---|
| `lowercase` | All characters must be lowercase after trim | Email addresses |
| `uppercase` | All characters must be uppercase after trim | Postal address components, ICD/CPT/HCPCS codes, LOINC/SNOMED/RxNorm/CVX/NDC/NPI (text), OMB/CDCREC race/ethnicity codes, short categorical status codes |
| `as_is` | No case transform — preserve source case after trim | Identifiers, phone numbers, provider/facility/payer names, free-text notes |
| `proper_case` | Title-case per dictionary entry (steward-approved) | BI display labels — only when steward has approved readability over source fidelity |
| `unknown` | AI could not determine case type with sufficient confidence | Requires steward classification before any transform is applied |

---

## Rule tags
| Tag | What it detects | Procedure | Action |
|---|---|---|---|
| `CASE_VIOLATION` | Values do not match the column's classified case rule | §1, §5 | Apply correct case transform in Silver ETL |
| `BLANKET_TRANSFORM_RISK` | All values are a single case but column type should be `as_is` or `proper_case` — suggests a blanket transform was applied | §2 | Investigate; revert to source if transform was undocumented |
| `CONFLICTING_RULE` | Column has values matching both uppercase and lowercase (mixed case at column level) | §2 | Resolve to one rule; document exception if intentional |
| `UNDOCUMENTED_CASE_TYPE` | AI returned `unknown` — case type cannot be determined heuristically | §3 | Steward must classify before any Silver transform |
| `EXCEPTION_NOT_DOCUMENTED` | Column deviates from the rule for its type but has no dictionary exception | §6 | Document exception in data dictionary or remediate |
| `PHI_NARRATIVE_CAPS_RISK` | Free-text / narrative PHI field has high ALL-CAPS rate | §4 | Review; avoid ALL CAPS for narrative PHI |

---

## Decision workflow
1. AI classifies each column into a case type from the §1 taxonomy.
2. Deterministic SQL measures compliance rate against that type.
3. Findings with `needs_steward_review = true` require a steward decision before action.
4. Engineer applies the decided transform in Silver ETL.
5. Re-run detector after fix to confirm resolved.

## Enforcement
- Wrong case on a typed field (e.g. mixed-case email in Gold, lowercase ICD-10) is a **certification defect** until remediated.
- Blanket table-level case transforms without a documented rule are **out of policy**.
- Inconsistent case on as-is display fields requires remediation or explicit dictionary exception before sign-off.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "case_formatting_detector"),
    "table_filter": _w("table_filter", ""),   # regex; empty = all tables
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer -- recorded on findings for lineage; also used to calibrate
    # exception tolerance (Bronze may retain source-faithful case)
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Sampling
    "sample_size": int(_w("sample_size", 10_000)),
    "seed":        int(_w("seed",        42)),

    # Detection thresholds
    # violation_threshold: % of non-null values that violate the case rule
    # before CASE_VIOLATION fires (set low -- any violation matters in certified layers)
    "violation_threshold":    float(_w("violation_threshold",    "2.0")),
    # caps_narrative_threshold: % of non-null values that are ALL CAPS before
    # PHI_NARRATIVE_CAPS_RISK fires on a narrative/as_is column
    "caps_narrative_threshold": float(_w("caps_narrative_threshold", "20.0")),
    # min_col_length: minimum average value length to include a column in case analysis
    # (avoids false positives on single-char flag columns)
    "min_col_length": float(_w("min_col_length", "3.0")),

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    # enable_ai: when True, uses ai_query() to classify each column into
    # exactly one case type from the §1 taxonomy.
    # When False, uses heuristic-only classification (less precise).
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "case_format_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run      : {RUN_ID}")
print(f"Scope    : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer    : {CFG['layer']}")
print(f"AI       : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results  : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Add detection logic in _check_case() in Cell 6
# ---------------------------------------------------------------------------

# ── Case type taxonomy (§1) ──────────────────────────────────────────────────
# These are the ONLY valid case types. AI must assign exactly one per column.
CASE_TYPES = {
    "lowercase",     # email addresses
    "uppercase",     # postal, ICD, CPT, HCPCS, LOINC, SNOMED, RxNorm, NDC, NPI-text,
                     # race/ethnicity codes, short categorical status codes
    "as_is",         # identifiers, phone, provider names, free-text notes
    "proper_case",   # BI display labels -- steward-approved only
    "unknown",       # AI could not determine; steward must classify
}

# ── Rule tags ────────────────────────────────────────────────────────────────
TAGS = {
    "CASE_VIOLATION":              "§Case-rules",
    "BLANKET_TRANSFORM_RISK":      "§Implementation",
    "CONFLICTING_RULE":            "§Implementation",
    "UNDOCUMENTED_CASE_TYPE":      "§Fields-without-rule",
    "EXCEPTION_NOT_DOCUMENTED":    "§Exceptions",
    "PHI_NARRATIVE_CAPS_RISK":     "§Sensitive-data",
}

SEVERITY: Dict[str, int] = {
    "CASE_VIOLATION":           10,
    "CONFLICTING_RULE":          8,
    "BLANKET_TRANSFORM_RISK":    7,
    "PHI_NARRATIVE_CAPS_RISK":   7,
    "EXCEPTION_NOT_DOCUMENTED":  6,
    "UNDOCUMENTED_CASE_TYPE":    5,
}

# ── Heuristic column name patterns ───────────────────────────────────────────
# Used as strong signals to the AI and as fallback when AI is disabled.
# AI classification takes precedence over all heuristics.

# lowercase: email address fields
RE_EMAIL = re.compile(
    r"(email|e_mail|e-mail|correo|mail_addr|emailaddr|email_address|"
    r"contact_email|member_email|patient_email)",
    re.I
)

# uppercase: postal address components
RE_POSTAL = re.compile(
    r"^(address|address_2|x_address|addr|street|city|state|zip|zip_code|"
    r"postal|postal_code|country|country_code|county|province|region)$",
    re.I
)

# uppercase: clinical / procedure / reference codes
RE_CODE = re.compile(
    r"(icd|cpt|hcpcs|drg|apc|loinc|snomed|rxnorm|rxcui|cvx|ndc|npi|"
    r"revenue_code|rev_code|proc_code|diag_code|dx_code|px_code|"
    r"race_code|ethnicity_code|gender_code|sex_code|claim_status|"
    r"member_status|enrollment_status|coverage_status|"
    r"iso_country|alpha2|alpha3)",
    re.I
)

# uppercase: short categorical / status codes (single-char or coded fields)
RE_CATEGORICAL = re.compile(
    r"(gender|sex|race|ethnicity|marital|status|type_cd|type_code|"
    r"flag|ind|indicator)$",
    re.I
)

# as_is: identifiers -- no case transform
RE_IDENTIFIER = re.compile(
    r"(_id$|_key$|_num$|_number$|_nbr$|_cd$|ssn|mrn|member_id|patient_id|"
    r"person_id|provider_id|context_id|record_id|source_id|data_source_id|"
    r"npi$|tin$|ein$|tax_id)",
    re.I
)

# as_is: free-text / narrative / notes
RE_NARRATIVE = re.compile(
    r"(note|notes|comment|comments|description|narrative|text|remark|"
    r"remarks|summary|detail|details|memo|observation|finding|reason|"
    r"explanation|history|chief_complaint|assessment|plan)",
    re.I
)

# as_is: provider / facility / payer names
RE_NAME = re.compile(
    r"(name|first_name|last_name|middle_name|full_name|display_name|"
    r"org_name|facility_name|provider_name|payer_name|group_name|"
    r"plan_name|employer_name|dba)",
    re.I
)

# as_is: phone numbers
RE_PHONE = re.compile(
    r"(phone|phone_number|fax|fax_number|cell|mobile|telephone)",
    re.I
)

def _heuristic_case_type(col: str) -> str:
    """
    Name-based heuristic case type. Used as a strong hint in the AI prompt
    and as fallback when AI is disabled. AI classification always takes precedence.
    """
    if RE_EMAIL.search(col):      return "lowercase"
    if RE_IDENTIFIER.search(col): return "as_is"
    if RE_PHONE.search(col):      return "as_is"
    if RE_NAME.search(col):       return "as_is"
    if RE_NARRATIVE.search(col):  return "as_is"
    if RE_POSTAL.search(col):     return "uppercase"
    if RE_CODE.search(col):       return "uppercase"
    if RE_CATEGORICAL.search(col):return "uppercase"
    return "unknown"

# ── Standardization rules ─────────────────────────────────────────────────────
STANDARDIZATION_RULES: Dict[str, list] = {

    "CASE_VIOLATION": [
        {"option_key": "apply_correct_case",
         "label":      "Apply documented case transform in Silver ETL",
         "sql":        "-- lowercase: LOWER(TRIM(col)) AS col\n"
                       "-- uppercase: UPPER(TRIM(col)) AS col\n"
                       "-- as_is:     TRIM(col) AS col  (no case change)",
         "notes":      "Apply exactly one rule per logical attribute (§1). "
                       "Case transform runs after trim and before domain validation (§2). "
                       "Wrong case on a typed field is a certification defect (§Enforcement)."},
        {"option_key": "document_exception",
         "label":      "Document exception in data dictionary if source case must be preserved",
         "sql":        "-- No transform. Flag column as 'do not transform' in data dictionary.",
         "notes":      "Fields requiring source-case preservation for audit must be flagged "
                       "'do not transform' in the dictionary with retention justification (§6). "
                       "Bronze may retain source-faithful case for replay (§6)."},
    ],

    "BLANKET_TRANSFORM_RISK": [
        {"option_key": "investigate_transform_history",
         "label":      "Investigate whether a blanket case transform was applied upstream",
         "sql":        "-- No transform. Audit pipeline history for undocumented UPPER()/LOWER() calls.",
         "notes":      "Blanket table-level case transforms without a documented rule are out of policy (§2). "
                       "If transform was applied, revert to source-faithful Bronze and re-apply per-column."},
    ],

    "CONFLICTING_RULE": [
        {"option_key": "resolve_to_single_rule",
         "label":      "Resolve to one case rule and apply consistently",
         "sql":        "-- Determine correct rule from §1 taxonomy and apply uniformly.",
         "notes":      "Do not chain conflicting rules on the same column (§2). "
                       "One logical attribute → one convention across all sources (§2)."},
        {"option_key": "document_intentional_mix",
         "label":      "Document intentional mixed-case as a dictionary exception",
         "sql":        "-- No transform. Document mixed-case rationale in data dictionary.",
         "notes":      "Use only when mix is intentional (e.g. proper nouns in a code field). "
                       "Exceptions require steward sign-off (§6)."},
    ],

    "UNDOCUMENTED_CASE_TYPE": [
        {"option_key": "steward_classify",
         "label":      "Steward classifies column into §1 taxonomy before any transform",
         "sql":        "-- No transform until case type is confirmed.",
         "notes":      "Fields without a listed rule default to as-is (§3). "
                       "To use proper case or another convention, document it in the data dictionary "
                       "before production load (§3). Do not upper/lower an entire table without a typed rule."},
    ],

    "EXCEPTION_NOT_DOCUMENTED": [
        {"option_key": "add_dictionary_entry",
         "label":      "Add exception entry to data dictionary",
         "sql":        "-- Document deviation from §1 rule in data dictionary with justification.",
         "notes":      "Exceptions (McDonald-style names, legal punctuation) must be documented "
                       "in mapping rules (§5). Inconsistent case on as-is display fields requires "
                       "remediation or explicit dictionary exception before sign-off (§Enforcement)."},
        {"option_key": "remediate_to_rule",
         "label":      "Remediate column to comply with §1 rule",
         "sql":        "-- Apply correct case transform in Silver ETL per §1 taxonomy.",
         "notes":      "Preferred if the deviation is a data quality issue, not an intentional exception."},
    ],

    "PHI_NARRATIVE_CAPS_RISK": [
        {"option_key": "review_narrative_caps",
         "label":      "Review narrative PHI field for ALL CAPS values; revert to source case",
         "sql":        "-- No transform. Investigate source. Avoid ALL CAPS for narrative PHI (§4).",
         "notes":      "Avoid ALL CAPS for narrative PHI/PII fields that may draw undue attention "
                       "in operational screens (§4). Coordinate with PHI storage and handling. "
                       "Coded fields (ICD, CPT) remain UPPERCASE per §1 -- this rule fires only "
                       "on narrative/free-text fields classified as as_is."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Case types: {sorted(CASE_TYPES)}")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema.
# Builds: tables, table_cols
# Only STRING / VARCHAR / CHAR columns are candidates for case analysis.
# NUMERIC, DATE, TIMESTAMP, BOOLEAN columns are excluded -- case rules don't apply.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

STRING_DTYPES_PFX = ("STRING", "VARCHAR", "CHAR")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    dt = r.data_type.upper()
    if any(dt.startswith(p) for p in STRING_DTYPES_PFX):
        table_cols[r.table_name].append((r.column_name, dt))

total_string_cols = sum(len(v) for v in table_cols.values())
print(f"Scope        : {cat}.{sch}")
print(f"Tables       : {len(tables)}")
print(f"String cols  : {total_string_cols}")
print(f"Layer        : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- key design decisions:
#
# 1. ONE SQL per table for ALL aggregate stats across ALL columns:
#    null_count, total, lower_count, upper_count, mixed_count,
#    avg_length, all_caps_count (for narrative PHI check).
#
# 2. Columns with avg_length < min_col_length are excluded (avoids
#    false positives on single-char flag columns that are legitimately
#    short coded values -- those are still included if name-heuristic
#    classifies them as uppercase categorical).
#
# 3. sample_df cached per table; unpersisted after the table loop.
#
# 4. ALL string columns are profiled -- case rules apply to every type.
#    The AI Classifier (Cell 5) then assigns the governing rule.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`")
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def profile_table_batch(tbl: str, cols: List[Tuple[str, str]]) -> Dict[str, dict]:
    """
    ONE SQL per table computing case distribution stats for ALL string columns.
    Returns dict keyed by column name.
    """
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    for col, dtype in cols:
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        v    = f"TRIM({cs})"   # always compare trimmed values

        exprs += [
            # null count
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            # avg length of non-null values (used to filter single-char columns)
            f"COALESCE(AVG(CASE WHEN {cs} IS NOT NULL THEN LENGTH(TRIM({cs})) END), 0) AS `__avglen__{safe}`",
            # lowercase: trimmed value equals LOWER(trimmed value), and trimmed value has alpha chars
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} = LOWER({v}) AND {v} RLIKE '[a-zA-Z]' THEN 1 END) AS `__lower__{safe}`",
            # uppercase: trimmed value equals UPPER(trimmed value), and has alpha chars
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} = UPPER({v}) AND {v} RLIKE '[a-zA-Z]' THEN 1 END) AS `__upper__{safe}`",
            # has_alpha: non-null, non-empty, contains at least one alpha character
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '[a-zA-Z]' THEN 1 END) AS `__alpha__{safe}`",
            # mixed: has alpha, is neither pure lower nor pure upper
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '[a-zA-Z]' "
            f"  AND {v} != LOWER({v}) AND {v} != UPPER({v}) THEN 1 END) AS `__mixed__{safe}`",
        ]

    try:
        row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Batch profile failed for {tbl}: {e}")
        return {}

    total = row["__total__"] or 0
    result = {}
    for col, dtype in cols:
        safe    = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        avg_len = float(row.get(f"__avglen__{safe}", 0) or 0)
        alpha   = int(row.get(f"__alpha__{safe}", 0) or 0)
        lower   = int(row.get(f"__lower__{safe}", 0) or 0)
        upper   = int(row.get(f"__upper__{safe}", 0) or 0)
        mixed   = int(row.get(f"__mixed__{safe}", 0) or 0)
        nulls   = int(row.get(f"__null__{safe}", 0) or 0)

        result[col] = {
            "dtype":           dtype,
            "total":           total,
            "null_count":      nulls,
            "alpha_count":     alpha,
            "lower_count":     lower,
            "upper_count":     upper,
            "mixed_count":     mixed,
            "avg_length":      avg_len,
            "heuristic_type":  _heuristic_case_type(col),
            "case_type":       "unknown",   # filled by AI Classifier
            "case_source":     "heuristic", # updated to "ai" after Cell 5
            "case_confidence": "low",
            "sample_vals":     [],
            "sample_violations": [],
        }

    return result


# ── Collect sample values ──────────────────────────────────────────────────
def collect_samples(tbl: str, col: str, sample_df: DataFrame,
                    case_type: str = "unknown") -> Tuple[list, list]:
    """Returns (sample_vals[:6], sample_violations[:5])."""
    cs = f"`{_esc(col)}`"
    try:
        sample_vals = [
            str(r["v"]) for r in
            sample_df.select(F.col(cs).alias("v"))
            .filter(F.col("v").isNotNull())
            .limit(6).collect()
        ]
    except Exception:
        sample_vals = []

    violations = []
    if case_type in ("lowercase", "uppercase") and sample_df is not None:
        try:
            if case_type == "lowercase":
                cond = F.col("v").isNotNull() & (F.trim(F.col("v")) != "") & \
                       (F.col("v").rlike("[a-zA-Z]")) & \
                       (F.trim(F.col("v")) != F.lower(F.trim(F.col("v"))))
            else:
                cond = F.col("v").isNotNull() & (F.trim(F.col("v")) != "") & \
                       (F.col("v").rlike("[a-zA-Z]")) & \
                       (F.trim(F.col("v")) != F.upper(F.trim(F.col("v"))))
            violations = [
                str(r["v"]) for r in
                sample_df.select(F.col(cs).alias("v"))
                .filter(cond).limit(5).collect()
            ]
        except Exception:
            violations = []
    return sample_vals, violations


# ── Run profiler ──────────────────────────────────────────────────────────────
table_profiles:  Dict[str, Dict[str, dict]] = {}
table_samples:   Dict[str, DataFrame]       = {}
total_profiled   = 0

for tbl in tables:
    cols      = table_cols.get(tbl, [])
    if not cols:
        continue
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    profiles = profile_table_batch(tbl, cols)

    # Filter out columns with avg_length below threshold UNLESS they have a
    # strong name-heuristic (categorical/code columns legitimately short)
    filtered = {}
    for col, info in profiles.items():
        keep = (
            info["alpha_count"] > 0 and (
                info["avg_length"] >= CFG["min_col_length"] or
                info["heuristic_type"] in ("uppercase", "lowercase")
            )
        )
        if keep:
            filtered[col] = info
            total_profiled += 1

    table_profiles[tbl] = filtered

print(f"Profiler done.")
print(f"  Columns with alpha content: {total_profiled} across {len(table_profiles)} table(s)")
for tbl, cols in table_profiles.items():
    print(f"  {tbl}: {len(cols)} column(s)")


In [0]:
# ---------------------------------------------------------------------------
# AI CASE CLASSIFIER -- scalable batch design:
#
# All ai_query() calls are chunked (BATCH_SIZE columns per call).
#
# One responsibility: classify each STRING column into exactly ONE case type
# from the §1 taxonomy:
#   lowercase | uppercase | as_is | proper_case | unknown
#
# Inputs to the AI per column:
#   - column name
#   - table name
#   - heuristic_type (name-based signal, strong prior)
#   - sample values (up to 6)
#   - lower_count / upper_count / mixed_count distribution from profiler
#
# The AI output is the authoritative case_type for the Rule Engine.
# Confidence is recorded; low-confidence columns are flagged for steward.
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


CASE_TYPE_INSTRUCTION = (
    "You are a healthcare data governance classifier. "
    "Assign exactly one case_type from this taxonomy to each column:\n"
    "  lowercase   -- email address fields only\n"
    "  uppercase   -- (a) postal address components: street, city, state, zip, country; "
    "(b) clinical/procedure codes: ICD, CPT, HCPCS, DRG, APC, revenue codes; "
    "(c) reference/lookup codes: LOINC, SNOMED, RxNorm, CVX, NDC, NPI stored as text, "
    "ISO country alpha-2/alpha-3; "
    "(d) short categorical/status codes: gender/sex/race/ethnicity codes, "
    "claim/enrollment/coverage status codes, OMB/CDCREC codes; "
    "(e) payer/plan/source abbreviation codes: short all-caps reference codes like "
    "MCR, MCD, BCB, UHC, PPO, HMO, EPO, SRC_A -- these are reference codes, not identifiers\n"
    "  as_is       -- (a) identifiers: columns with _id, _key, _num, _nbr suffix, "
    "SSN, MRN, NPI as numeric -- as_is even if values look like codes; "
    "(b) phone numbers; "
    "(c) provider/facility/payer NAMES (full legal names, display names -- not abbreviation codes); "
    "(d) free-text notes, narratives, comments, clinical observations\n"
    "  proper_case -- BI display labels ONLY when steward has approved readability over fidelity\n"
    "  unknown     -- genuinely ambiguous after considering all signals\n"
    "Classification rules -- apply in this priority order:\n"
    "(1) Identifiers (_id, _key, _num, _nbr suffix, ssn, mrn) are ALWAYS as_is. Never override.\n"
    "(2) Free-text/narrative fields (notes, comments, description, observation, assessment) "
    "are ALWAYS as_is. Never override.\n"
    "(3) Use the empirical case distribution as a strong signal:\n"
    "    - upper_count / alpha_count >= 0.85 AND avg_length <= 10: "
    "strongly suggests uppercase (code or abbreviation field).\n"
    "    - lower_count / alpha_count >= 0.85: strongly suggests lowercase or as_is.\n"
    "    - high mixed_count with avg_length > 15: strongly suggests as_is (name or narrative).\n"
    "(4) The 'prefer as_is when in doubt' tiebreaker applies ONLY when the distribution "
    "is genuinely mixed (no dominant signal from rule 3) AND the column name gives no "
    "clear indication. Do NOT apply it to override a strong empirical signal -- "
    "a column that is 85-100% uppercase with short values is very likely a code field.\n"
    "(5) Return unknown only when truly unable to classify after applying rules 1-4."
)


def classify_table(tbl: str) -> None:
    profiles = table_profiles.get(tbl, {})
    if not profiles:
        return

    if not CFG["enable_ai"]:
        # Heuristic-only: use name-based classification
        for col, info in profiles.items():
            info["case_type"]   = info["heuristic_type"]
            info["case_source"] = "heuristic"
            info["case_confidence"] = "medium" if info["heuristic_type"] != "unknown" else "low"
        return

    col_list = list(profiles.keys())
    for chunk in _chunked(col_list):
        payload = json.dumps([{
            "col":            col,
            "table":          tbl,
            "heuristic_type": profiles[col]["heuristic_type"],
            "samples":        profiles[col]["sample_vals"][:6],
            "lower_count":    profiles[col]["lower_count"],
            "upper_count":    profiles[col]["upper_count"],
            "mixed_count":    profiles[col]["mixed_count"],
            "alpha_count":    profiles[col]["alpha_count"],
            # avg_length feeds the empirical signal rules in the AI prompt:
            # short avg_length (<=10) + high upper_count strongly indicates a code field;
            # long avg_length (>15) + mixed case strongly indicates a name/narrative field.
            "avg_length":     round(profiles[col]["avg_length"], 1),
        } for col in chunk], default=str)

        prompt = (
            f"{CASE_TYPE_INSTRUCTION}\n"
            "Reply ONLY with a JSON array -- no prose, no markdown fences.\n"
            f"Columns to classify: {payload}\n"
            'Return: [{"col":"<name>","case_type":"<type>","confidence":"high|medium|low",'
            '"rationale":"<one sentence>"}]'
        )
        try:
            results = json.loads(_ai_query(prompt))
            for item in results:
                col = item.get("col", "")
                if col not in profiles:
                    continue
                ct = item.get("case_type", "unknown").lower().strip()
                if ct not in CASE_TYPES:
                    ct = "unknown"
                profiles[col].update({
                    "case_type":       ct,
                    "case_source":     "ai",
                    "case_confidence": item.get("confidence", "low"),
                    "ai_rationale":    item.get("rationale", ""),
                })
        except Exception as e:
            print(f"  [WARN] AI classification chunk failed for {tbl}: {e}")
            for col in chunk:
                if col in profiles:
                    profiles[col]["case_type"]       = profiles[col]["heuristic_type"]
                    profiles[col]["case_source"]      = "heuristic_fallback"
                    profiles[col]["case_confidence"]  = "low"
                    profiles[col]["ai_error"]         = str(e)

    # After classification, collect violation samples now that case_type is known
    sample_df = table_samples.get(tbl)
    for col, info in profiles.items():
        sv, viol = collect_samples(tbl, col, sample_df, info["case_type"])
        info["sample_vals"]       = sv
        info["sample_violations"] = viol


# ── Run classifier ────────────────────────────────────────────────────────────
for tbl in list(table_profiles.keys()):
    classify_table(tbl)
    types = {}
    for col, info in table_profiles[tbl].items():
        t = info["case_type"]
        types[t] = types.get(t, 0) + 1
    type_str = ", ".join(f"{t}:{n}" for t, n in sorted(types.items()))
    print(f"  ok {tbl}: {type_str}")

print(f"\nAI classification done.")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE:
# All stats come from the pre-computed profile dict.
# No additional full-table SQL runs here.
#
# One _check_case() function per column; fires findings based on:
#   - case_type from AI Classifier
#   - lower_count / upper_count / mixed_count from Profiler
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, info, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    # auto_decided_action is honoured whenever explicitly passed -- the caller
    # is responsible for only supplying it when the action is unambiguous
    # (e.g. high-confidence CASE_VIOLATION on a typed field per §1).
    # The previous len(options)==1 guard silently nullified it for rules like
    # CASE_VIOLATION that have multiple options but a clear default action.
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "data_type":                info.get("dtype", "UNKNOWN"),
        "layer":                    CFG["layer"],
        "case_type":                info.get("case_type", "unknown"),
        "case_source":              info.get("case_source", "heuristic"),
        "case_confidence":          info.get("case_confidence", "low"),
        "ai_rationale":             info.get("ai_rationale", ""),
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_case(tbl: str, col: str, info: dict) -> list:
    total      = info["total"]
    null_count = info["null_count"]
    alpha      = info["alpha_count"]
    lower      = info["lower_count"]
    upper      = info["upper_count"]
    mixed      = info["mixed_count"]
    case_type  = info["case_type"]
    confidence = info["case_confidence"]
    findings   = []

    if alpha == 0 or total == 0:
        return findings   # no alpha content to evaluate

    # ── UNDOCUMENTED_CASE_TYPE ──────────────────────────────────────────────
    # AI returned unknown -- steward must classify before any transform
    if case_type == "unknown":
        findings.append(_finding(tbl, col, info, "UNDOCUMENTED_CASE_TYPE",
            0, total, info["sample_vals"][:5],
            "Case type could not be determined for this column. "
            "Fields without a listed rule default to as-is (§3). "
            "Steward must classify into the §1 taxonomy before any Silver transform.",
            confidence="low",
        ))
        return findings   # no further checks without a known case type

    # ── CASE_VIOLATION ───────────────────────────────────────────────────────
    # For lowercase: violations = alpha values that are NOT pure lowercase
    # For uppercase: violations = alpha values that are NOT pure uppercase
    # For as_is / proper_case: no mandatory case -- no CASE_VIOLATION fires
    violation_count = 0
    violation_msg   = ""

    if case_type == "lowercase":
        # Violations: has alpha, not pure lowercase (= upper + mixed)
        violation_count = (upper + mixed)
        if violation_count > 0 and alpha > 0:
            viol_pct = violation_count / alpha * 100
            if viol_pct > CFG["violation_threshold"]:
                violation_msg = (
                    f"{violation_count:,} value(s) ({viol_pct:.1f}% of alpha values) "
                    "are not lowercase. Email addresses and other lowercase-typed fields "
                    "must be LOWER(TRIM(col)) in Silver (§1, §Enforcement)."
                )

    elif case_type == "uppercase":
        # Violations: has alpha, not pure uppercase (= lower + mixed)
        violation_count = (lower + mixed)
        if violation_count > 0 and alpha > 0:
            viol_pct = violation_count / alpha * 100
            if viol_pct > CFG["violation_threshold"]:
                violation_msg = (
                    f"{violation_count:,} value(s) ({viol_pct:.1f}% of alpha values) "
                    "are not uppercase. Clinical/postal/code-typed fields "
                    "must be UPPER(TRIM(col)) in Silver (§1, §Enforcement)."
                )

    if violation_msg:
        # Auto-decide only when AI confidence is high: the rule is prescriptive
        # (§1), the correct transform is unambiguous, and the only reason to keep
        # needs_steward_review=True at high confidence is if the classification
        # itself might be wrong -- which high confidence rules out.
        # At medium/low confidence the classification may be uncertain, so the
        # steward must confirm before any Silver transform is applied.
        _auto_action = "apply_correct_case" if confidence == "high" else None
        findings.append(_finding(tbl, col, info, "CASE_VIOLATION",
            violation_count, total,
            info["sample_violations"][:5],
            violation_msg,
            confidence="high" if confidence in ("high","medium") else "medium",
            auto_decided_action=_auto_action,
        ))

    # ── BLANKET_TRANSFORM_RISK ───────────────────────────────────────────────
    # as_is column where 100% of alpha values are a single case --
    # suggests an undocumented blanket UPPER/LOWER was applied upstream.
    #
    # EXEMPTION -- structured identifiers: columns classified as as_is because
    # RE_IDENTIFIER matched the name (member_id, patient_id, mrn, claim_id, etc.)
    # legitimately contain uppercase-prefixed codes like MBR-001, P001, CLM001.
    # Their uppercase content is part of the code format, not a transform artifact.
    # The procedure says identifiers are as_is with no case transform (§1) -- a
    # 100%-uppercase identifier column is expected, not suspicious.
    # Only flag when the as_is classification comes from a NAME or NARRATIVE
    # heuristic (provider_name, facility_name, notes) where a blanket UPPER()
    # would be a genuine policy violation.
    _is_identifier_col = bool(RE_IDENTIFIER.search(col))

    if case_type in ("as_is", "proper_case") and alpha >= 10 and not _is_identifier_col:
        lower_pct = lower / alpha * 100 if alpha else 0
        upper_pct = upper / alpha * 100 if alpha else 0
        if upper_pct == 100.0:
            findings.append(_finding(tbl, col, info, "BLANKET_TRANSFORM_RISK",
                upper, total, info["sample_vals"][:5],
                f"100% of alpha values in this as-is column are UPPERCASE. "
                "This may indicate an undocumented blanket UPPER() transform was applied. "
                "Provider/facility/payer names and free-text fields must not be globally "
                "uppercased without a documented rule (§2, §1). "
                "Audit pipeline history; revert to source-faithful Bronze if transform was undocumented.",
                confidence="medium",
            ))
        elif lower_pct == 100.0:
            findings.append(_finding(tbl, col, info, "BLANKET_TRANSFORM_RISK",
                lower, total, info["sample_vals"][:5],
                f"100% of alpha values in this as-is column are lowercase. "
                "This may indicate an undocumented blanket LOWER() transform was applied. "
                "Blanket table-level case transforms are out of policy (§2). "
                "Audit pipeline history; revert to source-faithful Bronze if transform was undocumented.",
                confidence="medium",
            ))

    # ── CONFLICTING_RULE ─────────────────────────────────────────────────────
    # lowercase or uppercase column with significant mixed-case contamination.
    # Only fires when CASE_VIOLATION did NOT already fire on the same column --
    # if CASE_VIOLATION fired, the violation count already captures the problem
    # and adding CONFLICTING_RULE on top produces redundant steward work items
    # for the same root cause. CONFLICTING_RULE is reserved for cases where
    # the mixed-case pattern is NOT simply "wrong case on a typed field" but
    # genuinely suggests two different conventions are in play simultaneously
    # (e.g. some rows uppercased by one source, others mixed-case by another).
    _case_violation_fired = any(f["classification"] == "CASE_VIOLATION" for f in findings)

    if case_type in ("lowercase", "uppercase") and alpha >= 10 and not _case_violation_fired:
        mixed_pct = mixed / alpha * 100 if alpha else 0
        if mixed_pct > CFG["violation_threshold"] * 2:
            findings.append(_finding(tbl, col, info, "CONFLICTING_RULE",
                mixed, total, info["sample_vals"][:5],
                f"{mixed:,} value(s) ({mixed_pct:.1f}%) are mixed-case in a "
                f"{'lowercase' if case_type == 'lowercase' else 'uppercase'}-typed column. "
                "Conflicting case conventions detected -- no single consistent rule "
                "is applied. Do not chain conflicting rules on the same column (§2). "
                "Resolve to one rule and apply consistently.",
                confidence="medium",
            ))

    # ── PHI_NARRATIVE_CAPS_RISK ──────────────────────────────────────────────
    # as_is narrative / free-text column with high ALL-CAPS rate (§4).
    # Identifiers are explicitly excluded: MBR-001, P001, MRN-001, CLM001 are
    # legitimately uppercase-prefixed codes whose format is intentional, not
    # a PHI narrative concern. §4 targets free-text notes and display names
    # (provider_name, facility_name, patient notes) where ALL CAPS may draw
    # undue attention in operational screens. Structured identifier codes are
    # out of scope for this rule per the procedure's §4 text and §1 exemption.
    if case_type == "as_is" and alpha >= 5 and not _is_identifier_col:
        upper_pct = upper / alpha * 100 if alpha else 0
        if upper_pct > CFG["caps_narrative_threshold"]:
            # Only flag name/narrative fields -- not generic as_is columns
            if RE_NARRATIVE.search(col) or RE_NAME.search(col) or info.get("heuristic_type") == "as_is":
                findings.append(_finding(tbl, col, info, "PHI_NARRATIVE_CAPS_RISK",
                    upper, total, info["sample_vals"][:5],
                    f"{upper_pct:.1f}% of values in this narrative/as-is field are ALL CAPS. "
                    "Avoid ALL CAPS for narrative PHI/PII fields that may draw undue attention "
                    "in operational screens (§4). Coded fields (ICD, CPT) remain UPPERCASE per §1 "
                    "-- this finding fires only on narrative/name fields.",
                    confidence="medium",
                ))

    # ── Low-confidence flag ──────────────────────────────────────────────────
    # When AI confidence is low and there is a violation, upgrade to steward review
    # (no extra finding -- the CASE_VIOLATION finding already has needs_steward_review=True).
    # When AI confidence is low and there are NO violations, flag for steward documentation --
    # EXCEPT for identifiers: _id/_key/_num/_nbr columns are explicitly defined in §1
    # as as_is with no case transform. Even a heuristic-fallback or low-confidence as_is
    # classification on them is correct by definition and needs no dictionary exception.
    # Flagging result_id, patient_id, claim_id etc. as EXCEPTION_NOT_DOCUMENTED is noise
    # that contradicts the procedure's clear exemption for identifiers.
    _low_conf_is_identifier = bool(RE_IDENTIFIER.search(col))
    # Also suppress when the name-based heuristic already agrees with the AI
    # classification: if _heuristic_case_type() matched the same type, that is
    # a strong prior independent of AI confidence (e.g. ethnicity_code heuristic
    # → uppercase, AI fallback → uppercase: the classification is right even at
    # low confidence, and the column may simply be compliant with no violations).
    _heuristic_agrees = info.get("heuristic_type", "unknown") == case_type
    if confidence == "low" and not findings and case_type not in ("unknown",)             and not _low_conf_is_identifier and not _heuristic_agrees:
        findings.append(_finding(tbl, col, info, "EXCEPTION_NOT_DOCUMENTED",
            0, total, info["sample_vals"][:5],
            f"Column classified as '{case_type}' with low confidence. "
            "Verify classification and document in data dictionary before production load (§3).",
            confidence="low",
        ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, cols in table_profiles.items():
    tbl_count = 0
    for col, info in cols.items():
        col_findings = _check_case(tbl, col, info)
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)
    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("case_type",                StringType(),  True),
    StructField("case_source",              StringType(),  True),
    StructField("case_confidence",          StringType(),  True),
    StructField("ai_rationale",             StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "CASE_VIOLATION",
    "CONFLICTING_RULE",
    "BLANKET_TRANSFORM_RISK",
}

if not all_findings:
    print("No case format findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking (certification defects)
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification defects): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","case_type","case_source",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","case_type")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 70)
    print("SECTION 3 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.countDistinct("column_name").alias("columns_affected"),
               F.collect_set("case_type").alias("case_types_present"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- Unknown / undocumented case types (steward action needed)
    unk_df = fdf.filter(F.col("classification") == "UNDOCUMENTED_CASE_TYPE")
    n_unk  = unk_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Undocumented case types ({n_unk}) -- steward classification required")
    print("  Fields without a listed rule default to as-is per §3")
    print("-" * 70)
    if n_unk:
        display(unk_df.select(
            "table_name","column_name","case_source","case_confidence",
            "ai_rationale","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","column_name"))
    else:
        print("  None.")

    # Section 5 -- PHI narrative caps
    phi_df = fdf.filter(F.col("classification") == "PHI_NARRATIVE_CAPS_RISK")
    n_phi  = phi_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- PHI narrative ALL CAPS risk ({n_phi})")
    print("  Avoid ALL CAPS for narrative PHI/PII fields (§4)")
    print("-" * 70)
    if n_phi:
        display(phi_df.select(
            "table_name","column_name","case_type","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","column_name"))
    else:
        print("  None.")

    # Section 6 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","case_type","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Columns analysed: {total_profiled}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Unknown type: {n_unk}  |  PHI caps: {n_phi}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
